# 03 — SVI and SSVI calibration

The SVI calibration minimizes weighted squared errors in total implied variance $w(k,T)=T\sigma_{\rm imp}^2(k,T)$. SSVI provides a global parameterization across maturities. This notebook fits the explicitly synthetic sample and plots the residuals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from volsurface.demo import synthetic_chain
from volsurface.cleaning import prepare_options
from volsurface.svi import fit_svi_by_expiry
from volsurface.ssvi import fit_ssvi
from volsurface.diagnostics import error_metrics
df=prepare_options(synthetic_chain())
params,fitted=fit_svi_by_expiry(df)
params.head()

In [ ]:
print("SVI in-sample IV errors:",error_metrics(fitted.iv,fitted.model_iv))
ssvi_params,theta_table,ssvi_fitted=fit_ssvi(df)
print("SSVI parameters:",ssvi_params)
print("SSVI in-sample IV errors:",error_metrics(ssvi_fitted.iv,ssvi_fitted.ssvi_iv))

In [ ]:
fig,ax=plt.subplots(figsize=(11,6))
for expiry,g in fitted.groupby("expiry_date"):
    g=g.sort_values("log_moneyness")
    ax.scatter(g.log_moneyness,100*g.iv,s=7,alpha=.5)
    ax.plot(g.log_moneyness,100*g.model_iv,linewidth=1.3,label=expiry)
ax.set(xlabel="Log-forward-moneyness",ylabel="Implied volatility (%)",title="Synthetic market marks vs SVI fit")
ax.grid(alpha=.25)
ax.legend(ncol=2)
plt.show()

## Interpretation and limitations

In-sample RMSE is a fit statistic, not a trading forecast. Raw SVI slices are not guaranteed to be free of cross-maturity calendar arbitrage; numerical diagnostics should be used as warnings, not proofs.